# Bölüm 1/7 — Veri Temizleme Ve Kesifsel Analiz



# Sosyal Medyada Dezenformasyon Tespiti ve İçerik Analizi
## Rusya-Ukrayna Savaşı Dezenformasyon Analizi 
1. Dezenformasyon Hangi Konularda Yayılıyor?
2. Dezenformasyon içeren metinler ile içermeyen metinlerin uzunlukları arasında bir fark var mı?
3. Dezenformasyon içeren metinlerde en sık hangi kelimeler kullanılıyor?
4. Yalan haberler ile doğru haberlerin duygu tonları arasında anlamlı bir fark var mı?
5. Tekli kelimelerden (unigram) ziyade, birbiriyle sık kullanılan ikili (bigram) ve üçlü (trigram) kelime gruplarını analiz etmek.
6. Dezenformasyonu tespit etmede makine öğrenmesi modelleri ne kadar başarılı?
7.  NLP modellerinin hangi kategorilerdeki dezenformasyonu saptamada daha çok zorlandığını belirlemek.
8.  Veri setinde yer alan etiket güvenilirlik (confidence) değerlerinin, modelin öğrenme süreciyle nasıl bir ilişkisi olduğunu incelemek.


## Kütüphaneler

Bu hücrede projede kullanılacak temel kütüphaneler içe aktarılır: veri işleme için `pandas`/`numpy`, metin temizliği için `re`, görselleştirme için `matplotlib` ve makine öğrenmesi için `scikit-learn` modülleri. Kod tekrarını azaltmak amacıyla, projede ilerleyen bölümlerde kullanılan tüm model sınıfları ve metrik fonksiyonları da bu hücrede tek seferde içe aktarılmıştır.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay, roc_curve, precision_recall_curve, auc,
    average_precision_score
)


## Veri setini yükleme

Veri seti (`veri_seti_duzenlendi.xlsx`) `pandas` ile okunur ve ilk birkaç satırı görüntülenerek sütun yapısı kontrol edilir.

In [ ]:
df = pd.read_excel("veri_seti_duzenlendi.xlsx")
df.head()

## Genel yapı ve eksik değer kontrolü¶

Veri setinin boyutu, sütun isimleri ve eksik değer durumu kontrol edilir. Bu adım, temizlik yapmadan önce veri kalitesi hakkında genel bir fikir edinmek için standart bir ilk kontroldür.

In [ ]:
print("Boyut:", df.shape)
print("\nSütunlar:", list(df.columns))
print("\nEksik değerler:\n", df.isnull().sum())
print("\nTekrarlı post_id sayısı:", df["post_id"].duplicated().sum())
print("Tekrarlı metin sayısı:", df["text"].duplicated().sum())

## Sınıf ve kategori dağılımları

`is_disinformation` (dezenformasyon etiketi) ve `news_type` (haber kategorisi) sütunlarının dağılımı incelenir; bu, veri setinin ne kadar dengeli/dengesiz olduğunu baştan görmek için önemlidir. **Not:** Bu veri setinde sınıflar dengesiz dağılmaktadır (dezenformasyon etiketli "yes" kayıtları çoğunluktadır); bu durum ilerleyen bölümlerde model değerlendirme metriği seçimini (Macro F1, "no" recall/precision takibi) doğrudan etkilemektedir.

In [ ]:
print("is_disinformation dağılımı:\n", df["is_disinformation"].value_counts(dropna=False))
print("\nnews_type dağılımı:\n", df["news_type"].value_counts(dropna=False))
print("\nconfidence dağılımı:\n", df["confidence"].value_counts(dropna=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 8))

df["is_disinformation"].value_counts().plot(kind="bar", ax=axes[0], color=["#4c72b0", "#c44e52"])
axes[0].set_title("is_disinformation dağılımı")
axes[0].set_xlabel("")
axes[0].set_ylabel("kayıt sayısı")
axes[0].bar_label(axes[0].containers[0])

df["news_type"].value_counts().plot(kind="bar", ax=axes[1], color="#55a868")
axes[1].set_title("news_type dağılımı")
axes[1].set_xlabel("")
axes[1].bar_label(axes[1].containers[0])

df["confidence"].value_counts().plot(kind="bar", ax=axes[2], color="#dd8452")
axes[2].set_title("confidence dağılımı")
axes[2].set_xlabel("")
axes[2].bar_label(axes[2].containers[0])

plt.tight_layout()
plt.show()

yes_orani = (df["is_disinformation"] == "yes").mean()
print(f"\n'yes' (dezenformasyon) sınıfının oranı: %{yes_orani*100:.1f} -- veri seti dengesiz.")


## Bozuk karakter (encoding) tespiti

Ham `text` sütununda, bozuk karakter kodlamasına (mojibake) işaret eden tipik karakter dizileri (â, ðŸ, Ã, vb.) aranarak kaç kayıtta bu sorunun bulunduğu tespit edilir.

In [ ]:
bozuk = df[df["text"].str.contains(r"â|ðŸ|Ã|�", regex=True, na=False)]
print("Bozuk karakter şüphesi olan kayıt sayısı:", len(bozuk))

## Metin temizleme — mojibake düzeltme ve `clean_text` oluşturma


`fix_mojibake()` fonksiyonu tanımlanır: UTF-8 baytlarının yanlışlıkla cp1252 kod sayfasıyla okunmasından kaynaklanan karakter bozulmasını, metni cp1252 ile yeniden kodlayıp UTF-8 olarak çözerek düzeltir. Ayrıca markdown/URL/emoji temizliği yapan yardımcı fonksiyonlar (`clean_text_base`, `clean_text_from_base`) da burada tanımlanıp uygulanır.

In [ ]:
def fix_mojibake(text):
    """UTF-8 metnin cp1252 olarak yanlış okunmasından kaynaklanan bozulmayı düzeltir.
    Not (YENİ): errors="ignore" kullanıldığı için encode/decode zaten exception fırlatmaz;
    önceki try/except bloğu hiçbir zaman tetiklenmeyen ölü kod olduğu için kaldırıldı."""
    return text.encode("cp1252", errors="ignore").decode("utf-8", errors="ignore")

def remove_emoji(text):
    """Emoji ve sembol/piktogram bloklarındaki unicode karakterleri kaldırır."""
    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001FAFF"  # semboller, piktogramlar, emoji
        "\U00002600-\U000027BF"  # çeşitli semboller & dingbats
        "\U0001F1E6-\U0001F1FF"  # bölgesel gösterge sembolleri (bayrak harfleri)
        "\U00002190-\U000021FF"  # oklar
        "\U0000FE0F"              # varyasyon seçici (emoji render modu)
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', text)

def remove_non_latin(text):
    """ASCII dışındaki karakterleri (Kiril, Çince, vb. script'ler) kaldırır.
    SADECE clean_text (vectorization) için kullanılır -- dil tespiti için değil,
    çünkü dil tespitinin yabancı script'i görebilmesi gerekir (bkz. üstteki not)."""
    return re.sub(r'[^\x00-\x7F]+', ' ', text)

def clean_text_base(text):
    """Dil tespiti için kullanılacak ara temizlik: mojibake + markdown/URL + emoji +
    görünmez karakterler. Latin-olmayan script'ler BİLEREK korunur."""
    t = fix_mojibake(text)
    t = re.sub(r'!\[.*?\]\(.*?\)', ' ', t)          # markdown görsel
    t = re.sub(r'\[([^\]]*)\]\([^)]*\)', r'\1', t)   # markdown link -> sadece etiket metni kalsın
    t = re.sub(r'http\S+|www\.\S+', ' ', t)          # URL
    t = re.sub(r'[*_`#>~]+', ' ', t)                 # markdown biçimlendirme karakterleri
    t = remove_emoji(t)
    t = re.sub(r'[\u200b\u200e\u200f\ufeff]', ' ', t)  # görünmez unicode karakterler
    t = re.sub(r'\s+', ' ', t).strip()
    t = t.lower()
    return t

def clean_text_from_base(base_text):
    """clean_text_base çıktısı üzerine non-Latin script temizliği uygular --
    vectorization için kullanılacak son hal."""
    t = remove_non_latin(base_text)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df_clean = df.copy()
df_clean["text_for_lang"] = df_clean["text"].apply(clean_text_base)
df_clean["clean_text"] = df_clean["text_for_lang"].apply(clean_text_from_base)

print("Temizlik öncesi örnek:")
print(df_clean["text"].iloc[1][:150])
print("\nTemizlik sonrası örnek (clean_text):")
print(df_clean["clean_text"].iloc[1][:150])

kalan_bozuk = df_clean["clean_text"].str.contains(r"â|ðŸ|Ã|�", regex=True, na=False).sum()
print("\nTemizlik sonrası hâlâ bozuk görünen kayıt sayısı:", kalan_bozuk)


## Yabancı script/alfabe tespiti ve tamamen yabancı dilde kalan kayıtların çıkarılması

Metindeki alfabetik karakterlerin ne kadarının Latin-dışı bir script'e (Kiril, Çince vb.) ait olduğunu ölçen `non_latin_ratio()` fonksiyonu tanımlanır. `langdetect` kütüphanesinin kısa/gazetecilik üslubundaki İngilizce başlıklarda yanlış dil tahmini yapması nedeniyle bu özel script-tabanlı yöntem tercih edilmiştir; %16 eşik değeri, gerçek yabancı-dil içerik ile İngilizce metin içindeki tekil yabancı kelimeleri net biçimde ayırmaktadır. **Sınırlılık:** Bu eşik değeri istatistiksel bir yöntemle değil, veri setindeki birkaç örnek üzerinde gözlemsel/deneysel olarak belirlenmiştir; farklı bir veri setinde yeniden kalibre edilmesi gerekebilir.

In [ ]:

# Gerçek veri üzerinde test ederken şunu bulduk: langdetect, kısa/farklı üsluplu
# İngilizce haber başlıklarında (örn. "cnn: uk delivers long-range missiles to
# ukraine." -> yanlışlıkla "no" / Norveççe) onlarca yanlış pozitif üretiyordu --
# kısa-metin eşiği bile bunu tam çözemedi (130 -> 118, hâlâ çoğu yanlış pozitif).
#
# Buna karşılık, veri setindeki GERÇEK yabancı-dil kirliliğini (örn. BBC Çince
# haberleri) doğrudan script/alfabe bazlı bir oranla çok daha güvenilir şekilde
# yakalayabiliyoruz: metnin alfabetik karakterlerinin ne kadarı Latin-olmayan
# (Kiril, Çince, vb.)? Test ettiğimizde net bir ayrım bulduk:
#   - Bilinen yanlış-pozitifler (gerçek İngilizce metinler): oran = 0.0
#   - Kanal hashtag'i olarak tek bir Kiril kelime içeren İngilizce metinler
#     (örn. "#ЦПД_інформує: Russia uses American historian...")     : oran ~0.11-0.13
#   - Gerçek yabancı-dil belgeler (BBC Çince haberleri)              : oran 0.63-0.79
# %15 eşiği bu ikisini net şekilde ayırıyor ve harici bir kütüphaneye
# (langdetect) bağımlılığı da ortadan kaldırıyor.

NON_LATIN_RATIO_THRESHOLD = 0.16

def non_latin_ratio(text):
    """Metindeki alfabetik karakterlerin ne kadarının Latin-olmayan (Kiril,
    Çince, vb.) script'e ait olduğunu 0-1 arası bir oran olarak döndürür."""
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    non_latin = [c for c in letters if ord(c) > 0x2FF]  # Latin + Latin-1 Ek bloğu dışı
    return len(non_latin) / len(letters)

# Not: "text_for_lang" kullanıyoruz (non-Latin script'ler korunmuş hali) --
# "clean_text" değil, çünkü clean_text'te bu script'ler zaten temizlenmiş olurdu.
df_clean["non_latin_ratio"] = df_clean["text_for_lang"].apply(non_latin_ratio)
df_clean["detected_lang"] = np.where(
    df_clean["non_latin_ratio"] > NON_LATIN_RATIO_THRESHOLD, "non-latin-script", "en"
)

print("Tespit dağılımı:\n", df_clean["detected_lang"].value_counts())

non_en = df_clean[df_clean["detected_lang"] != "en"]
print("\nLatin-olmayan script şüpheli kayıt sayısı:", len(non_en))
non_en[["post_id", "clean_text", "non_latin_ratio"]].sort_values("non_latin_ratio", ascending=False)


Tespit edilen yabancı-script kayıtlar (`detected_lang != "en"`) veri setinden çıkarılır ve öncesi/sonrası kayıt sayısı karşılaştırılır.

In [ ]:
oncesi = df_clean.shape[0]
df_clean = df_clean[df_clean["detected_lang"] == "en"].reset_index(drop=True)
print("Dil filtresi öncesi:", oncesi, "-> sonrası:", df_clean.shape[0], "  (çıkarılan:", oncesi - df_clean.shape[0], ")")

**Sızıntı kontrolü:** Dil filtresiyle çıkarılan kayıtların (`non_en`) `is_disinformation` ve `news_type` dağılımı, kalan veri setiyle karşılaştırılır. Amaç, bu filtrenin belirli bir sınıfı veya kategoriyi sistematik olarak hedef almadığını (yani modele istemsizce bir sinyal sızdırmadığını) doğrulamaktır.

In [ ]:
non_en_oran = (non_en["is_disinformation"] == "yes").mean()
genel_oran = (df_clean["is_disinformation"] == "yes").mean()

print(f"Genel veri setinde 'yes' oranı: {genel_oran:.3f}")
print(f"Dil filtresiyle çıkarılan kayıtlarda 'yes' oranı: {non_en_oran:.3f}")

print("\nÇıkarılan kayıtların news_type dağılımı:")
print(non_en["news_type"].value_counts(normalize=True).round(3))

print("\nGenel veri setinin news_type dağılımı:")
print(df_clean["news_type"].value_counts(normalize=True).round(3))


## Konu dışı (alakasız) haberlerin tespiti ve temizlenmesi

Rusya-Ukrayna savaşıyla alakasız olabilecek (İsrail/Gazze, İran/Suriye, Tayvan, Afganistan, Covid, Kosova gibi) konulardaki kayıtlar, anahtar kelime eşleştirmesiyle tespit edilir. Ukrayna/Rusya ile ilişkili terim içerenler (gerçekte konuyla ilgili olabileceğinden) korunur, geri kalanlar veri setinden çıkarılır.

In [ ]:
# Rusya-Ukrayna savasiyla alakasiz olabilecek haberler icin aday kelime gruplari
keywords = [
    "israel", "israeli", "lebanon", "lebanese",
    "hezbollah", "gaza", "hamas", "palestine", "palestinian",
    "syria", "syrian", "iran", "iranian",
    "taiwan", "taiwanese", "afghanistan", "afghan", "libya",
    "covid", "coronavirus", "kosovo"
]
pattern = "|".join(keywords)
adaylar = df_clean[df_clean["clean_text"].str.contains(pattern, case=False, na=False, regex=True)].copy()
print("Aday kayit sayisi:", len(adaylar))

# Icinde Ukrayna-Rusya ile ilgili anahtar kelime gecenleri haric tut (konuyla ilgili olabilirler)
ukraine_russia_keywords = [
    "ukraine", "ukrainian", "ukrainians", "russia", "russian", "russians",
    "putin", "zelensky", "zelenskiy", "kyiv", "kiev", "kremlin", "moscow",
    "donbas", "crimea", "nato"
]
pattern_ur = "|".join(ukraine_russia_keywords)

# --- Grup 1: Israil/Lubnan/Gazze ---
israel_group = ["israel", "israeli", "lebanon", "lebanese", "hezbollah", "gaza", "hamas", "palestine", "palestinian"]
pattern_israel = "|".join(israel_group)
israel_aday = adaylar[adaylar["clean_text"].str.contains(pattern_israel, case=False, na=False, regex=True)].copy()
israel_irrelevant_candidates = israel_aday[
    ~israel_aday["clean_text"].str.contains(pattern_ur, case=False, na=False, regex=True)
].copy()
print("Israil/Lubnan/Gazze grubu aday:", len(israel_aday), "-> alakasiz:", len(israel_irrelevant_candidates))

# --- Grup 2: Iran/Suriye/Taiwan/Afganistan/Libya/Covid/Kosova ---
# Bu grup elle incelendi (iran_syria_inceleme.xlsx); sadece asagidaki 2 kayit
# gercekten Rusya-Ukrayna savasiyla ilgili bulundu (Iran drone/Suriye-Wagner baglami),
# geri kalan kayitlar alakasiz kabul edildi.
other_keywords = ["syria", "syrian", "iran", "iranian", "taiwan", "taiwanese",
                   "afghanistan", "afghan", "libya", "covid", "coronavirus", "kosovo"]
pattern_other = "|".join(other_keywords)
# Not (YENİ) isimlendirme netligi: "other_candidates" burada zaten Ukrayna-Rusya
# kelimesi GECMEYEN (yani zaten "alakasiz aday" olan) kayitlari tutuyor -- "candidates"
# ismi henuz karar verilmemis gibi izlenim veriyor olabilir, bunu belirtmek icin not dustum.
other_aday = adaylar[adaylar["clean_text"].str.contains(pattern_other, case=False, na=False, regex=True)].copy()
other_candidates = other_aday[
    ~other_aday["clean_text"].str.contains(pattern_ur, case=False, na=False, regex=True)
].copy()  # zaten "alakasiz" olan alt kume -- manuel_alakali_id ile birazi geri eklenecek

manuel_alakali_id = ["telegram_rybar_32598", "telegram_rybar_40577"]
other_irrelevant_candidates = other_candidates[~other_candidates["post_id"].isin(manuel_alakali_id)].copy()
print("Iran/Suriye/... grubu aday:", len(other_candidates), "-> alakasiz:", len(other_irrelevant_candidates))

# --- Alakasiz kayitlari veri setinden cikar ---
irrelevant_all = pd.concat([israel_irrelevant_candidates, other_irrelevant_candidates])
oncesi = df_clean.shape[0]
df_clean = df_clean.drop(index=irrelevant_all.index.unique())
print("\nÖnceki boyut:", oncesi)
print("Yeni boyut:", df_clean.shape[0])
print("Silinen kayit:", oncesi - df_clean.shape[0])

**Sızıntı kontrolü:** Konu-dışı filtresiyle çıkarılan kayıtların (`irrelevant_all`) `is_disinformation` oranı ve `news_type` dağılımı, genel veri setiyle karşılaştırılır. Çıkarılan kayıtlardaki "yes" oranının genel orandan çok sapmaması, bu filtrenin dezenformasyon etiketiyle karıştırılan bir örüntü yakalamadığını (yani konu-alaka filtresinin dolaylı bir sınıf filtresine dönüşmediğini) gösterir.

In [ ]:
irrelevant_oran = (irrelevant_all["is_disinformation"] == "yes").mean()
genel_oran_2 = (df["is_disinformation"] == "yes").mean()

print(f"Genel veri setinde 'yes' oranı: {genel_oran_2:.3f}")
print(f"Konu-dışı filtresiyle çıkarılan kayıtlarda 'yes' oranı: {irrelevant_oran:.3f}")

print("\nÇıkarılan kayıtların news_type dağılımı:")
print(irrelevant_all["news_type"].value_counts(normalize=True).round(3))


## Çok kısa kalan metinlerin temizlenmesi

`clean_text` uzunluğu 25 karakterin altında kalan (genellikle kaynakta zaten kırpılmış gelen) kayıtlar tespit edilip veri setinden çıkarılır; bu kayıtlar anlamlı bir sınıflandırma sinyali taşımayacak kadar kısadır.

In [ ]:
kisa_metin_mask = df_clean["clean_text"].str.len() < 25
silinenler = df_clean[kisa_metin_mask].copy()

print("Silinecek cok kisa/bos kayit sayisi:", kisa_metin_mask.sum())
df_clean = df_clean[~kisa_metin_mask].reset_index(drop=True)
print("Kisa metinler cikarildiktan sonra veri seti boyutu:", df_clean.shape)

print("\nSilinen kayitlar:")
silinenler[["post_id", "clean_text", "is_disinformation", "news_type"]]

In [ ]:
kisa_oran = (silinenler["is_disinformation"] == "yes").mean()
genel_oran_3 = (df_clean["is_disinformation"] == "yes").mean()  # not: df_clean burada filtre SONRASI hali

print(f"Genel veri setinde 'yes' oranı: {genel_oran_3:.3f}")
print(f"Çok kısa metin filtresiyle çıkarılan kayıtlarda 'yes' oranı: {kisa_oran:.3f}")

print("\nÇıkarılan kayıtların news_type dağılımı:")
print(silinenler["news_type"].value_counts(normalize=True).round(3))

print("\nGenel veri setinin news_type dağılımı:")
print(df_clean["news_type"].value_counts(normalize=True).round(3))

## Bu bölümü kaydet
Bir sonraki bölümün bu noktadan devam edebilmesi için tüm oturum (değişkenler, modeller, fonksiyonlar) diske kaydedilir.

In [ ]:
import dill
dill.dump_session('checkpoint_1.pkl')
print('Oturum checkpoint_1.pkl olarak kaydedildi.')